In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import os

from catboost import CatBoostRegressor

In [3]:
train_df = pd.read_csv("../data/raw/train.csv")
test_df = pd.read_csv("../data/raw/test.csv")


In [4]:
train_df['RoadType'] = (
    train_df['RoadType']
    .fillna("MISSING")
    .astype(str)
)

test_df['RoadType'] = (
    test_df['RoadType']
    .fillna("MISSING")
    .astype(str)
)

train_df['geohash'] = (
    train_df['geohash']
    .fillna("MISSING")
    .astype(str)
)

test_df['geohash'] = (
    test_df['geohash']
    .fillna("MISSING")
    .astype(str)
)

In [6]:
test_df['Temperature'] = (
    test_df['Temperature']
    .fillna(
        train_df['Temperature'].median()
    )
)

test_df['NumberofLanes'] = (
    test_df['NumberofLanes']
    .fillna(
        train_df['NumberofLanes'].median()
    )
)

In [7]:
features = [
    'geohash',
    'RoadType'
]

In [8]:
X_train = train_df[features]

y_train = train_df['demand']

In [9]:
model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    random_seed=42,
    verbose=100
)

In [10]:
model.fit(
    X_train,
    y_train,
    cat_features=[
        'geohash',
        'RoadType'
    ]
)

0:	learn: 0.1378887	total: 50.6ms	remaining: 50.6s
100:	learn: 0.0575148	total: 376ms	remaining: 3.35s
200:	learn: 0.0566773	total: 712ms	remaining: 2.83s
300:	learn: 0.0563586	total: 1.05s	remaining: 2.43s
400:	learn: 0.0561854	total: 1.39s	remaining: 2.07s
500:	learn: 0.0560205	total: 1.73s	remaining: 1.73s
600:	learn: 0.0558844	total: 2.08s	remaining: 1.38s
700:	learn: 0.0557794	total: 2.43s	remaining: 1.04s
800:	learn: 0.0556852	total: 2.78s	remaining: 692ms
900:	learn: 0.0556077	total: 3.14s	remaining: 345ms
999:	learn: 0.0555370	total: 3.47s	remaining: 0us


CatBoostRegressor(depth=6, iterations=1000, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

In [11]:
X_test = test_df[features]

test_pred = model.predict(
    X_test
)

In [12]:
test_pred = np.clip(
    test_pred,
    0,
    1
)

In [14]:
submission = pd.DataFrame({
    'Index': test_df['Index'],
    'demand': test_pred
})

In [19]:
os.makedirs("../data/submissions", exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

filename = f"../data/submissions/submission_{timestamp}.csv"

submission.to_csv(
    filename,
    index=False
)

print(f"Saved to {filename}")

Saved to ../data/submissions/submission_20260601_233749.csv
